# GRIDPILOT AI — Member 2 — Task 2
## Demand Data Ingestion Pipeline (Exploration)

**Purpose:** Prototype and validate the ingestion/cleaning logic (schema validation, timestamp normalization, duplicate/gap detection, missing-value handling, range validation) before porting it into services/data/**. Does not train any model.


## 0. Setup

In [ ]:
!pip -q install pandas numpy pyarrow

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
print("Libraries loaded.")

## 1. Load canonical raw dataset

Use the SAME columns identified in Task 1 (`docs/data/openstef-demand.md`).
Do not re-guess column names here — paste in the confirmed ones.

In [ ]:
DATA_PATH = "REPLACE_WITH_ACTUAL_PATH.csv"

# Confirmed from Task 1 inspection (docs/data/openstef-demand.md)
TIMESTAMP_COL = None   # e.g. "timestamp"
DEMAND_COL = None      # e.g. "actual_load"
ZONE_COL = None        # e.g. "zone_id"

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Rows:", len(df), "Columns:", len(df.columns))
df.head()

## 2. Schema validation

Fail loudly if required columns are missing or of the wrong type.

In [ ]:
REQUIRED_COLUMNS = [c for c in [TIMESTAMP_COL, DEMAND_COL] if c is not None]

def validate_schema(data, required_cols):
    missing = [c for c in required_cols if c not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    print("Schema OK. Required columns present:", required_cols)

validate_schema(df, REQUIRED_COLUMNS)

## 3. Normalize timestamps and sort

In [ ]:
df["_ts"] = pd.to_datetime(df[TIMESTAMP_COL], errors="coerce")

invalid_ts = df["_ts"].isna().sum()
print("Invalid/unparseable timestamps:", invalid_ts)

df = df.sort_values("_ts").reset_index(drop=True)
print("Sorted by timestamp. Range:", df["_ts"].min(), "to", df["_ts"].max())

## 4. Detect duplicate timestamps

In [ ]:
dupe_key = ["_ts"] + ([ZONE_COL] if ZONE_COL else [])
dup_mask = df.duplicated(subset=dupe_key, keep=False)
duplicate_count = df.duplicated(subset=dupe_key).sum()

print("Duplicate rows (by", dupe_key, "):", duplicate_count)
if duplicate_count:
    display(df[dup_mask].sort_values(dupe_key).head(20))

## 5. Detect missing intervals (gaps)

Assumes a fixed sampling frequency confirmed in Task 1. Replace `EXPECTED_FREQ_MINUTES`.

In [ ]:
EXPECTED_FREQ_MINUTES = None  # e.g. 15 -- set from Task 1 findings

def find_gaps(ts_series, expected_minutes):
    ts_series = ts_series.dropna().drop_duplicates().sort_values()
    deltas = ts_series.diff().dropna()
    expected = pd.Timedelta(minutes=expected_minutes)
    gaps = deltas[deltas != expected]
    return gaps

if EXPECTED_FREQ_MINUTES is not None:
    gaps = find_gaps(df["_ts"], EXPECTED_FREQ_MINUTES)
    print("Number of irregular intervals:", len(gaps))
    if len(gaps):
        display(gaps.describe().to_frame("gap_stats"))
else:
    print("Set EXPECTED_FREQ_MINUTES from the Task 1 frequency analysis first.")

## 6. Handle missing demand values explicitly

Do not silently interpolate. Decide and document the rule.

In [ ]:
missing_demand = df[DEMAND_COL].isna().sum()
print("Missing demand values:", missing_demand, f"({missing_demand / len(df) * 100:.3f}%)")

# Explicit policy (edit before running in production):
# Option A: leave as NaN and let the feature pipeline handle gaps
# Option B: forward-fill within a small max gap (document the max gap size)
# Option C: drop rows with missing demand
MISSING_DEMAND_POLICY = "leave_as_nan"  # placeholder -- must be a documented decision

print("Policy in effect:", MISSING_DEMAND_POLICY)

## 7. Validate physical numeric ranges

Flag, don't silently drop, until the range rule is confirmed against source documentation.

In [ ]:
demand_numeric = pd.to_numeric(df[DEMAND_COL], errors="coerce")

MIN_PLAUSIBLE = None  # set from domain knowledge / Task 1 stats
MAX_PLAUSIBLE = None

out_of_range = pd.Series([False] * len(df))
if MIN_PLAUSIBLE is not None:
    out_of_range |= (demand_numeric < MIN_PLAUSIBLE)
if MAX_PLAUSIBLE is not None:
    out_of_range |= (demand_numeric > MAX_PLAUSIBLE)

print("Out-of-range demand rows:", int(out_of_range.sum()))
if out_of_range.any():
    display(df.loc[out_of_range, [TIMESTAMP_COL, DEMAND_COL]].head(20))

## 8. Produce cleaned canonical dataset

In [ ]:
canonical_cols = ["_ts", DEMAND_COL] + ([ZONE_COL] if ZONE_COL else [])
canonical_df = df[canonical_cols].rename(columns={"_ts": "timestamp"}).copy()

OUTPUT_DIR = Path("./data_ingestion_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

canonical_path = OUTPUT_DIR / "canonical_demand_dataset.parquet"
canonical_df.to_parquet(canonical_path, index=False)
print("Saved canonical dataset:", canonical_path, "rows:", len(canonical_df))

## 9. Data-quality report

In [ ]:
quality_report = {
    "source_file": DATA_PATH,
    "rows_in": int(len(df)),
    "rows_out": int(len(canonical_df)),
    "invalid_timestamps": int(invalid_ts),
    "duplicate_rows": int(duplicate_count),
    "missing_demand_count": int(missing_demand),
    "missing_demand_pct": float(missing_demand / len(df) * 100),
    "out_of_range_demand_count": int(out_of_range.sum()),
    "missing_demand_policy": MISSING_DEMAND_POLICY,
}

with open(OUTPUT_DIR / "data_quality_report.json", "w") as f:
    json.dump(quality_report, f, indent=2, default=str)

print(json.dumps(quality_report, indent=2, default=str))

## Next step

Once this logic is verified here, port it into `services/data/**` as the production ingestion pipeline. Do not train a model in this notebook — that is Chunk 4/5.